In [91]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import joblib
import pandas as pd
import re
import numpy as np
import os
import re

df = pd.read_csv('../data/fitfood_products_full93.csv')
df.head()

,категория,название,ссылка,калории,белки,жиры,углеводы
0,Готовая еда,49.Пирожок печеный Вак бэлиш с мясом и рисом 100г,https://magnit.ru/product/1000027714-pirozhok_...,260.0,11.4,6.1,38.9
1,Готовая еда,99.Пирожок печёный Элеш с начинкой из мяса кур...,https://magnit.ru/product/1000127914-pirozhok_...,200.0,5.6,NaN,32.2
2,Готовая еда,21.Сдоба Смаковница 100г,https://magnit.ru/product/1000022170-sdoba_sma...,290.0,7.4,8.0,47.5
3,Готовая еда,69.Сдоба с изюмом ХК Лавина 250г,https://magnit.ru/product/1000010649-sdoba_s_i...,350.0,7.0,8.0,62.0
4,Готовая еда,39.Лаваш Кавказский 350г4.8· 34 отзыва,https://magnit.ru/product/1000095996-lavash_ka...,250.0,8.5,1.0,52.5


In [92]:
def clean_title(title):
    if not title or not isinstance(title, str):
        return ""
    
    text = str(title)
    
    text = re.sub(r'^\d+\.', '', text)
    text = re.sub(r'\d+\.\d+\s*[·\-]?\s*\d+\s*отзывов?', '', text)
    text = re.sub(r'\d+\.\d+\s*[·\-]?\s*\d+\s*отзыва', '', text)
    text = re.sub(r'нет\s*отзывов', '', text)
    text = re.sub(r'\d+\.\d+', '', text)
    text = re.sub(r'\d+\s*[·]\s*\d+\s*отзывов?', '', text)
    text = re.sub(r'⭐\s*\d+\.\d+', '', text)
    text = re.sub(r'\d+[\d\s]*[₽руб]', '', text)
    text = re.sub(r'\+5%\s*с\s*премиум', '', text)
    text = re.sub(r'-\d+%', '', text)
    text = re.sub(r'в\s*корзину', '', text)
    text = re.sub(r'финальная\s*цена', '', text)
    text = re.sub(r'товар\s*18\+', '', text)
    text = re.sub(r'\d+\s*[гкл]\s*$', '', text)
    text = re.sub(r'\d+\.?\d*\s*[гклмл]', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.lstrip('.')
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip().lower()

df['clean_title'] = df['название'].apply(clean_title)
df.head()

,категория,название,ссылка,калории,белки,жиры,углеводы,clean_title
0,Готовая еда,49.Пирожок печеный Вак бэлиш с мясом и рисом 100г,https://magnit.ru/product/1000027714-pirozhok_...,260.0,11.4,6.1,38.9,пирожок печеный вак бэлиш с мясом и рисом
1,Готовая еда,99.Пирожок печёный Элеш с начинкой из мяса кур...,https://magnit.ru/product/1000127914-pirozhok_...,200.0,5.6,NaN,32.2,пирожок печёный элеш с начинкой из мяса курицы...
2,Готовая еда,21.Сдоба Смаковница 100г,https://magnit.ru/product/1000022170-sdoba_sma...,290.0,7.4,8.0,47.5,сдоба смаковница
3,Готовая еда,69.Сдоба с изюмом ХК Лавина 250г,https://magnit.ru/product/1000010649-sdoba_s_i...,350.0,7.0,8.0,62.0,сдоба с изюмом хк лавина
4,Готовая еда,39.Лаваш Кавказский 350г4.8· 34 отзыва,https://magnit.ru/product/1000095996-lavash_ka...,250.0,8.5,1.0,52.5,лаваш кавказский


In [96]:
categories = {
    'Молочный прилавок': ['молоко', 'кефир', 'йогурт', 'творог', 'сметана', 'ряженка', 'сливки', 'масло', 'сыр'],
    'Мясо, птица, рыба': ['курица', 'говядина', 'свинина', 'фарш', 'колбаса', 'сосиска', 'семга', 'минтай', 'филе', 'окорочок'],
    'Овощи и фрукты': ['помидор', 'огурец', 'капуста', 'морковь', 'лук', 'чеснок', 'картофель', 'яблоко', 'банан', 'апельсин', 'груша', 'мандарин'],
    'Хлеб и выпечка': ['хлеб', 'батон', 'лаваш', 'булочка', 'пирожок', 'сдоба', 'багет'],
    'Бакалея': ['гречка', 'рис', 'макароны', 'крупа', 'мука', 'овсянка', 'перловка', 'пшено'],
    'Вода и напитки': ['сок', 'вода', 'лимонад', 'квас', 'компот', 'морс'],
    'Сладости': ['шоколад', 'конфета', 'печенье', 'торт', 'пирожное', 'вафли', 'кекс', 'мармелад'],
    'Замороженные продукты': ['пельмени', 'вареники', 'котлета', 'блины', 'овощи замороженные'],
    'Консервы': ['тушенка', 'горошек', 'кукуруза', 'фасоль', 'шпроты'],
    'Готовая еда': ['салат', 'оливье', 'суп', 'борщ', 'пюре', 'пицца', 'бургер', 'роллы'],
    'Чай, кофе, какао': ['чай', 'кофе', 'какао', 'цикорий'],
    'Снеки': ['чипсы', 'сухарики', 'орешки', 'семечки', 'попкорн']
}

brands = ['Простоквашино', 'Савушкин', 'Вкуснотеево', 'Моя цена', 'Каждый день']
weights = ['200г', '400г', '500г', '800г', '1кг', '250г', '300г', '100г']
reviews = ['', '4.8·128 отзывов', '4.5·45 отзывов', 'Нет отзывов']

kbju_db = {
    'молоко': {'калории': 52, 'белки': 3, 'жиры': 3.2, 'углеводы': 4.7},
    'творог': {'калории': 120, 'белки': 18, 'жиры': 5, 'углеводы': 3},
    'курица': {'калории': 165, 'белки': 20, 'жиры': 9, 'углеводы': 0},
    'хлеб': {'калории': 260, 'белки': 8, 'жиры': 3, 'углеводы': 48},
    'гречка': {'калории': 340, 'белки': 13, 'жиры': 3.5, 'углеводы': 68},
    'яблоко': {'калории': 52, 'белки': 0.3, 'жиры': 0.2, 'углеводы': 14},
    'шоколад': {'калории': 550, 'белки': 7, 'жиры': 35, 'углеводы': 50},
    'пельмени': {'калории': 220, 'белки': 12, 'жиры': 8, 'углеводы': 25},
    'сок': {'калории': 45, 'белки': 0.5, 'жиры': 0.2, 'углеводы': 11}
}

def generate_kbju(product_name):
    for key in kbju_db:
        if key in product_name.lower():
            return kbju_db[key]
    return {'калории': random.randint(50, 400), 'белки': round(random.uniform(1, 15), 1), 
            'жиры': round(random.uniform(1, 20), 1), 'углеводы': round(random.uniform(10, 60), 1)}

def generate_products(n_per_category=300):
    data = []
    product_id = 1
    
    for category, words in categories.items():
        for i in range(n_per_category):
            word = random.choice(words)
            brand = random.choice(brands) if random.random() > 0.3 else ''
            weight = random.choice(weights)
            review = random.choice(reviews)
            
            if brand:
                name = f"{word} {brand} {weight} {review}".strip()
            else:
                name = f"{word} {weight} {review}".strip()
            
            name = re.sub(r'\s+', ' ', name)
            kbju = generate_kbju(word)
            
            data.append({
                'категория': category,
                'название': name,
                'ссылка': f'generated_{product_id}',
                'калории': kbju['калории'],
                'белки': kbju['белки'],
                'жиры': kbju['жиры'],
                'углеводы': kbju['углеводы']
            })
            product_id += 1
    
    return pd.DataFrame(data)

df_generated = generate_products(300)
print(f"Сгенерировано: {len(df_generated)} товаров")
print(df_generated['категория'].value_counts())

df_original = pd.read_csv('../data/fitfood_products_full93.csv', encoding='utf-8-sig')
df_combined = pd.concat([df_original, df_generated], ignore_index=True)
print(f"\nОбъединено: {len(df_combined)} товаров")
df_combined.to_csv('../data/fitfood_products_full_augmented.csv', index=False, encoding='utf-8-sig')
df = df_combined

def clean_title(title):
    if not title or not isinstance(title, str):
        return ""
    
    text = str(title).lower()
    
    text = re.sub(r'^\d+\.', '', text)
    text = re.sub(r'\d+\.\d+\s*[·\-]?\s*\d+\s*отзывов?', '', text)
    text = re.sub(r'нет\s*отзывов', '', text)
    text = re.sub(r'\d+\.\d+', '', text)
    text = re.sub(r'\d+[\d\s]*[₽руб]', '', text)
    text = re.sub(r'\+5%\s*с\s*премиум', '', text)
    text = re.sub(r'в\s*корзину', '', text)
    text = re.sub(r'товар\s*18\+', '', text)
    text = re.sub(r'\d+\s*[гкл]\s*$', '', text)
    text = re.sub(r'\d+\.?\d*\s*[гклмл]', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.lstrip('.')
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

df['clean_title'] = df['название'].apply(clean_title)
df['title_length'] = df['clean_title'].apply(len)
df['word_count'] = df['clean_title'].apply(lambda x: len(x.split()))

print("clean_title создана")
keywords = {
    'молоко': 'Молочный прилавок', 'кефир': 'Молочный прилавок', 'творог': 'Молочный прилавок',
    'сыр': 'Молочный прилавок', 'йогурт': 'Молочный прилавок', 'масло': 'Молочный прилавок',
    'сметана': 'Молочный прилавок', 'ряженка': 'Молочный прилавок',
    'курица': 'Мясо, птица, рыба', 'куриное': 'Мясо, птица, рыба', 'филе': 'Мясо, птица, рыба',
    'говядина': 'Мясо, птица, рыба', 'свинина': 'Мясо, птица, рыба', 'фарш': 'Мясо, птица, рыба',
    'колбаса': 'Мясо, птица, рыба', 'сосиска': 'Мясо, птица, рыба', 'индейка': 'Мясо, птица, рыба',
    'окорочок': 'Мясо, птица, рыба', 'голень': 'Мясо, птица, рыба', 'утка': 'Мясо, птица, рыба',
    'семга': 'Рыба, морепродукты', 'сёмга': 'Рыба, морепродукты', 'лосось': 'Рыба, морепродукты',
    'минтай': 'Рыба, морепродукты', 'треска': 'Рыба, морепродукты', 'форель': 'Рыба, морепродукты',
    'горбуша': 'Рыба, морепродукты', 'кижуч': 'Рыба, морепродукты',
    'хлеб': 'Хлеб и выпечка', 'лаваш': 'Хлеб и выпечка', 'батон': 'Хлеб и выпечка',
    'булочка': 'Хлеб и выпечка', 'пирожок': 'Хлеб и выпечка', 'сдоба': 'Хлеб и выпечка',
    'гречка': 'Бакалея', 'рис': 'Бакалея', 'макароны': 'Бакалея', 'крупа': 'Бакалея',
    'мука': 'Бакалея', 'овсянка': 'Бакалея', 'горох': 'Бакалея', 'чечевица': 'Бакалея',
    'яблоко': 'Овощи и фрукты', 'яблоки': 'Овощи и фрукты', 'банан': 'Овощи и фрукты',
    'апельсин': 'Овощи и фрукты', 'груша': 'Овощи и фрукты', 'мандарин': 'Овощи и фрукты',
    'помидор': 'Овощи и фрукты', 'помидоры': 'Овощи и фрукты', 'огурец': 'Овощи и фрукты',
    'картофель': 'Овощи и фрукты', 'капуста': 'Овощи и фрукты', 'морковь': 'Овощи и фрукты',
    'лук': 'Овощи и фрукты', 'чеснок': 'Овощи и фрукты', 'лимон': 'Овощи и фрукты',
    'шоколад': 'Сладости', 'печенье': 'Сладости', 'конфета': 'Сладости',
    'торт': 'Сладости', 'пирожное': 'Сладости', 'вафли': 'Сладости',
    'пельмени': 'Замороженные продукты', 'вареники': 'Замороженные продукты', 'котлета': 'Замороженные продукты',
    'блины': 'Замороженные продукты',
    'сок': 'Вода и напитки', 'вода': 'Вода и напитки', 'лимонад': 'Вода и напитки',
    'квас': 'Вода и напитки', 'компот': 'Вода и напитки', 'морс': 'Вода и напитки',
    'чай': 'Чай, кофе, какао', 'кофе': 'Чай, кофе, какао', 'какао': 'Чай, кофе, какао',
    'чипсы': 'Снеки', 'сухарики': 'Снеки', 'орешки': 'Снеки', 'семечки': 'Снеки', 'попкорн': 'Снеки',
    'салат': 'Готовая еда', 'суп': 'Готовая еда', 'борщ': 'Готовая еда', 'пюре': 'Готовая еда',
    'пицца': 'Готовая еда', 'бургер': 'Готовая еда', 'роллы': 'Готовая еда',
    'тушенка': 'Консервы', 'горошек': 'Консервы', 'кукуруза': 'Консервы', 'фасоль': 'Консервы'
}

keywords_df = pd.DataFrame()
for word in keywords.keys():
    keywords_df[f'has_{word}'] = df['clean_title'].str.contains(word, na=False).astype(int)

df = pd.concat([df, keywords_df], axis=1)

vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),
    stop_words=['г', 'кг', 'мл', 'л', 'шт', 'и', 'в', 'на', 'с', 'по', 'из']
)

X_tfidf = vectorizer.fit_transform(df['clean_title']).toarray()
X_keywords = keywords_df.values
X_length = df[['title_length', 'word_count']].values

features = np.hstack([X_tfidf, X_keywords, X_length])

print(f"Признаки: {features.shape}")

le = LabelEncoder()
y_encoded = le.fit_transform(df['категория'])

X_train, X_test, y_train, y_test = train_test_split(
    features, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Обучающая: {len(X_train)}")
print(f"Тестовая: {len(X_test)}")

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочность: {accuracy:.2%}")

print("\nОтчет по категориям:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/xgb_model.pkl')
joblib.dump(vectorizer, 'models/tfidf_vectorizer.pkl')
joblib.dump(le, 'models/label_encoder.pkl')
joblib.dump(keywords, 'models/keywords.pkl')

print("\nМодель сохранена")

test_products = [
    "Молоко деревенское 3.2%",
    "Куриное филе охлаждённое",
    "Сёмга слабосолёная",
    "Яблоки красные",
    "Лосось гриль",
    "Индейка запеченная"
]
print("\nТЕСТ:")
for product in test_products:
    cleaned = clean_title(product)
    
    fallback = None
    for word, cat in keywords.items():
        if word in cleaned:
            fallback = cat
            break
    
    vec = vectorizer.transform([cleaned]).toarray()
    kw_features = [1 if word in cleaned else 0 for word in keywords.keys()]
    features_test = np.hstack([vec, [kw_features], [[len(cleaned), len(cleaned.split())]]])
    
    pred = le.inverse_transform([model.predict(features_test)[0]])[0]
    proba = max(model.predict_proba(features_test)[0]) * 100
    
    if fallback and proba < 60:
        print(f"{product} -> {fallback} (fallback, модель: {pred} {proba:.0f}%)")
    else:
        print(f"{product} -> {pred} ({proba:.0f}%)")

Сгенерировано: 3600 товаров
категория
Молочный прилавок        300
Мясо, птица, рыба        300
Овощи и фрукты           300
Хлеб и выпечка           300
Бакалея                  300
Вода и напитки           300
Сладости                 300
Замороженные продукты    300
Консервы                 300
Готовая еда              300
Чай, кофе, какао         300
Снеки                    300
Name: count, dtype: int64

Объединено: 11300 товаров
clean_title создана
Признаки: (11300, 2094)
Обучающая: 9040
Тестовая: 2260
[0]	validation_0-mlogloss:2.34051
[100]	validation_0-mlogloss:0.67069
[200]	validation_0-mlogloss:0.48985
[300]	validation_0-mlogloss:0.40972
[400]	validation_0-mlogloss:0.37037
[499]	validation_0-mlogloss:0.34801

Точность: 91.15%

Отчет по категориям:
                       precision    recall  f1-score   support

              Бакалея       0.82      0.94      0.88       259
       Вода и напитки       0.94      0.97      0.95       238
          Готовая еда       0.91      0.81